In [4]:
# Cell 1 — Install dependencies
# !pip install chromadb sentence-transformers pypdf tiktoken

In [5]:
# Cell 2 — Imports
import os
import time
import textwrap

import chromadb
import tiktoken
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


In [6]:
# Cell 3 — Configuration settings
CHUNK_SIZES   = [256, 512, 1024]   # token sizes to compare
CHUNK_OVERLAP = 30                  # overlapping tokens between chunks
TOP_K         = 3                   # number of results to retrieve
EMBED_MODEL   = "all-MiniLM-L6-v2" # embedding model
CHROMA_PATH   = "./chroma_db"       # local folder to store ChromaDB
ENCODING_NAME = "cl100k_base"       # tiktoken encoding

print("✅ Config ready")
print(f"   Chunk sizes : {CHUNK_SIZES}")
print(f"   Overlap     : {CHUNK_OVERLAP} tokens")
print(f"   Top-K       : {TOP_K}")
print(f"   Model       : {EMBED_MODEL}")

✅ Config ready
   Chunk sizes : [256, 512, 1024]
   Overlap     : 30 tokens
   Top-K       : 3
   Model       : all-MiniLM-L6-v2


---
##   Load PDF
Upload your PDF using the Jupyter file browser (click the ⬆️ Upload button on the left panel),  
then set the filename below.

In [7]:
# Cell 4 — Load PDF
# ✏️  CHANGE THIS to your PDF filename
PDF_PATH = "Vikas.pdf"

def extract_text_from_pdf(pdf_path):
    """Extract all text from a PDF file."""
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        text = page.extract_text()
        if text:
            full_text += text + "\n"
    return full_text

# Load the PDF
if os.path.exists(PDF_PATH):
    full_text = extract_text_from_pdf(PDF_PATH)
    print(f"✅ PDF loaded: {PDF_PATH}")
    print(f"   Pages      : {len(PdfReader(PDF_PATH).pages)}")
    print(f"   Characters : {len(full_text):,}")
    print(f"\n📄 Preview (first 500 chars):")
    print("-" * 60)
    print(full_text[:500])
else:
    print(f"❌ File not found: '{PDF_PATH}'")
    print("   Please upload your PDF and update PDF_PATH above.")

✅ PDF loaded: Vikas.pdf
   Pages      : 1
   Characters : 2,168

📄 Preview (first 500 chars):
------------------------------------------------------------
SAI VIKAS RACHAKONDA
Karimnagar, Telangana
saivikasrachakonda@gmail.com ⋄ +91-9494205343 ⋄ LinkedIn ⋄ GitHub
OBJECTIVE
IT undergraduate with strong foundations in Java, SQL, and backend development using Spring Boot.Gained Hands-on
experience in NLP-based projects and building basic LLM-powered applications.
SKILLS AND INTERESTS
Programming Languages: Java, Python, SQL
AI/ML: LLMs, NLP Techniques, Model Evaluation, Generative AI
F rontend: HTML, CSS, JavaScript, React.js, TailwindCSS
Backend: Sp


---
##Chunking the text
Split the full text into chunks of 256, 512, and 1024 tokens.

In [8]:
# Cell 5 — Define chunking function
def chunk_by_tokens(text, chunk_size, overlap=CHUNK_OVERLAP):
    """Split text into chunks of `chunk_size` tokens with overlap."""
    enc    = tiktoken.get_encoding(ENCODING_NAME)
    tokens = enc.encode(text)
    chunks = []
    step   = chunk_size - overlap
    for i in range(0, len(tokens), step):
        chunk_tokens = tokens[i : i + chunk_size]
        if len(chunk_tokens) < 10:   # skip tiny trailing fragments
            continue
        chunks.append(enc.decode(chunk_tokens))
    return chunks

print("✅ Chunking function defined")

✅ Chunking function defined


In [9]:
# Cell 6 — Chunk at all 3 sizes and compare counts
chunks_256  = chunk_by_tokens(full_text, 256)
chunks_512  = chunk_by_tokens(full_text, 512)
chunks_1024 = chunk_by_tokens(full_text, 1024)

print("✅ Chunking complete")
print()
print(f"  {'Chunk Size':<12} {'# Chunks':<10} {'Avg Chars/Chunk'}")
print(f"  {'-'*40}")
for size, chunks in [(256, chunks_256), (512, chunks_512), (1024, chunks_1024)]:
    avg = int(sum(len(c) for c in chunks) / len(chunks))
    print(f"  {size:<12} {len(chunks):<10} {avg}")

✅ Chunking complete

  Chunk Size   # Chunks   Avg Chars/Chunk
  ----------------------------------------
  256          3          792
  512          2          1146
  1024         1          2168


---
##  Setup ChromaDB & Load Embedding Model

In [10]:
# Cell 7 — Initialise ChromaDB and load embedding model
client = chromadb.PersistentClient(path=CHROMA_PATH)
print(f"✅ ChromaDB ready at: {CHROMA_PATH}")

print(f"⏳ Loading embedding model: {EMBED_MODEL} ...")
model = SentenceTransformer(EMBED_MODEL)
print(f"✅ Model loaded")

✅ ChromaDB ready at: ./chroma_db
⏳ Loading embedding model: all-MiniLM-L6-v2 ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 327.44it/s]


✅ Model loaded


---
## Ingest Chunks into ChromaDB
Creates one collection per chunk size.

In [11]:
# Cell 8 — Build ChromaDB collections for each chunk size
def build_collection(client, model, text, chunk_size):
    """Chunk text, embed it, and store in a ChromaDB collection."""
    name = f"pdf_chunks_{chunk_size}"

    # Drop existing collection (fresh run)
    try:
        client.delete_collection(name)
    except Exception:
        pass

    collection = client.create_collection(
        name=name,
        metadata={"hnsw:space": "cosine"}
    )

    chunks     = chunk_by_tokens(text, chunk_size)
    t0         = time.time()
    embeddings = model.encode(chunks, show_progress_bar=False).tolist()
    ids        = [f"c{i}" for i in range(len(chunks))]
    metadatas  = [{"chunk_index": i, "chunk_size": chunk_size} for i in range(len(chunks))]

    collection.add(documents=chunks, embeddings=embeddings, ids=ids, metadatas=metadatas)
    elapsed = time.time() - t0

    print(f"  ✅ chunk_size={chunk_size:>5}  →  {len(chunks):>4} chunks  ({elapsed:.1f}s)")
    return {"collection": collection, "chunk_size": chunk_size, "n_chunks": len(chunks)}


print("⏳ Building collections ...")
print()
collections_info = []
for cs in CHUNK_SIZES:
    info = build_collection(client, model, full_text, cs)
    collections_info.append(info)

print()
print("✅ All collections built and stored in ChromaDB")

⏳ Building collections ...



  ✅ chunk_size=  256  →     3 chunks  (0.9s)
  ✅ chunk_size=  512  →     2 chunks  (0.3s)
  ✅ chunk_size= 1024  →     1 chunks  (0.3s)

✅ All collections built and stored in ChromaDB


In [12]:
# Cell 9 — Set your query here
# ✏️  CHANGE THIS to test different questions
QUERY = "What is the main topic discussed in this document?"

print(f"🔍 Query: {QUERY}")

🔍 Query: What is the main topic discussed in this document?


In [13]:
# Cell 10 — Run retrieval on all 3 collections
def query_collection(collection, model, query, top_k=TOP_K):
    """Query a collection and return ranked results with similarity scores."""
    q_emb   = model.encode([query]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=top_k)
    out = []
    for i, (doc, dist) in enumerate(zip(results["documents"][0], results["distances"][0])):
        out.append({"rank": i + 1, "score": round(1 - dist, 4), "text": doc})
    return out


all_results = []
for info in collections_info:
    hits = query_collection(info["collection"], model, QUERY)
    all_results.append({
        "chunk_size": info["chunk_size"],
        "n_chunks":   info["n_chunks"],
        "results":    hits,
    })

print("✅ Retrieval complete")

✅ Retrieval complete


---
#Compare Results

In [14]:
# Cell 11 — Print detailed results per chunk size
print("=" * 72)
print(f"  QUERY: {QUERY}")
print("=" * 72)

for entry in all_results:
    cs   = entry["chunk_size"]
    hits = entry["results"]
    nc   = entry["n_chunks"]

    print(f"\n┌── Chunk size: {cs} tokens  │  {nc} total chunks")
    for hit in hits:
        bar     = "█" * int(hit["score"] * 20)
        preview = textwrap.fill(hit["text"].strip(), width=65,
                                initial_indent="   ", subsequent_indent="   ")
        print(f"│  Rank {hit['rank']}  similarity = {hit['score']:.4f}  {bar}")
        print(preview)
        print(f"│  {'─' * 68}")
    print("└")

  QUERY: What is the main topic discussed in this document?

┌── Chunk size: 256 tokens  │  3 total chunks
│  Rank 1  similarity = 0.2044  ████
   2022 – 2026 CGPA: 8.03 / 10 Intermediate — Sr Junior College
   2020 – 2022 MPC – 981/1000 marks INTERNSHIP EXPERIENCE F
   rontend W eb Developer Aug 2025 – Oct 2025 Edunet Foundation –
   AICTE Internship • Worked on frontend components using React
   and JavaScript as part of an AICTE-certified internship.
   PROJECTS PLABA – Plain Language Adaptation of Biomedical
   Abstracts Jan 2025 – Jan 2026 • Developed an NLP-based system
   to simplify complex biomedical research abstracts with
   linguistic clarity. • Applied domain-specific transformer
   models (BioBART, BioGPT, SciFive) using T5 and BART
   architectures. Mobile Phone Price Prediction Oct 2024 – Dec
   202 • Analyzed a dataset of 10,000 entries to forecast mobile
   market values with 89% accuracy. • Conducted exploratory data
   analysis (EDA) to identify key price drivers su

In [15]:
# Cell 12 — Summary comparison table
print()
print("=" * 72)
print("  SUMMARY TABLE — Chunk Size Comparison")
print("=" * 72)
print(f"  {'Chunk Size':>12} │ {'# Chunks':>9} │ {'Avg Similarity':>14} │ Observation")
print(f"  {'─'*12}─┼─{'─'*9}─┼─{'─'*14}─┼─{'─'*25}")

observations = {
    256:  "Precise, narrow context",
    512:  "Balanced — often best ✅",
    1024: "Broad context, more noise",
}

for entry in all_results:
    cs     = entry["chunk_size"]
    nc     = entry["n_chunks"]
    scores = [r["score"] for r in entry["results"]]
    avg    = sum(scores) / len(scores) if scores else 0
    obs    = observations.get(cs, "")
    print(f"  {cs:>12} │ {nc:>9} │ {avg:>14.4f} │ {obs}")

print()
print("  💡 Higher similarity = better semantic match to your query")
print("  💡 Also read the retrieved text — scores alone don't tell everything")


  SUMMARY TABLE — Chunk Size Comparison
    Chunk Size │  # Chunks │ Avg Similarity │ Observation
  ─────────────┼───────────┼────────────────┼──────────────────────────
           256 │         3 │         0.1566 │ Precise, narrow context
           512 │         2 │         0.1123 │ Balanced — often best ✅
          1024 │         1 │         0.1713 │ Broad context, more noise

  💡 Higher similarity = better semantic match to your query
  💡 Also read the retrieved text — scores alone don't tell everything
